**Projet d'examen — Data Collection** :
Web scraping, nettoyage de données et déploiement d'une application Streamlit

**Compréhension du projet d’examen**

Ce projet d’examen a pour objectif de mettre en œuvre une chaîne complète de collecte, de traitement, de stockage et de valorisation de données issues du Web. Il consiste à collecter des données provenant de deux sources distinctes, Books to Scrape et Gaaraas, en utilisant deux approches complémentaires : une approche coding avec Selenium, suivie du nettoyage des données, et une approche no-code avec l’extension Web Scraper, permettant de conserver les données brutes. Les données nettoyées seront ensuite stockées dans une base de données SQL et exploitées à travers une application Streamlit déployée, offrant des fonctionnalités de scraping sur plusieurs pages, de téléchargement des données brutes et de visualisation des données nettoyées sous forme de dashboard. Enfin, l’application intégrera deux formulaires d’évaluation réalisés respectivement avec Kobo et Google Forms, afin de recueillir l’appréciation des utilisateurs. Le projet permet ainsi de couvrir l’ensemble du processus allant de la collecte des données à leur exploitation dans une application web interactive.

Rappel des sources

Source 1: Books to Scrape : https://books.toscrape.com/catalogue/page-1.html

Source 2: Gaaraas (annonces auto Dakar) : https://www.gaaraas.com/fr/users/dakar-auto?page=2

In [1]:
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 56.6 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


# Importation des packages

In [2]:
# importer packages
import pandas as pd
from selenium.webdriver.common.by import By
import google_colab_selenium as gs
import time

# Collecte des données avec Selenium

In [3]:
# Lancer le navigateur
driver = gs.Chrome()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
url = 'https://books.toscrape.com/catalogue/page-1.html'

driver.get(url)

In [5]:
# Identifier les conteneurs des livres
containers = driver.find_elements(By.CSS_SELECTOR, 'article.product_pod')

# Vérifier le nombre de livres trouvés sur la page
len(containers)

20

In [6]:
# Sélectionner le premier livre de la liste des conteneurs
container = containers[0]

# Récupération du lien vers la fiche détaillée du livre

In [7]:
# Récupérer l'URL de la fiche détaillée du livre
lien = container.find_element(
    By.CSS_SELECTOR, 'h3 a'
).get_attribute('href')

print(lien)

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [8]:
driver.get(lien)

# Extraction de la description du livre

In [9]:
# Extraire la description située après la section Product Description
description = driver.find_element(
    By.XPATH,
    "//div[@id='product_description']/following-sibling::p[1]"
).text

print(description)

It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?Baby, I think someone down here'sGot it in for you. Shel, you never sounded

In [10]:
# Créer un DataFrame final vide
df_final = pd.DataFrame()

# Parcourir les pages du catalogue
for i in range(1, 3):

    # Construire l'URL de chaque page
    url = f'https://books.toscrape.com/catalogue/page-{i}.html'

    # Ouvrir la page
    driver.get(url)

    # Identifier les conteneurs des livres
    containers = driver.find_elements(
        By.CSS_SELECTOR,
        'article.product_pod'
    )

    # Compter le nombre de produits sur la page
    nombre_produits = len(containers)

    # Créer une liste vide pour stocker les données de la page
    data = []
        # Liste temporaire pour mémoriser les informations des livres
    livres = []

    # Parcourir les conteneurs de la page catalogue
    for container in containers:

        try:
            # Extraire les informations disponibles sur la page catalogue
            titre = container.find_element(
                By.CSS_SELECTOR, 'h3 a'
            ).get_attribute('title')

            prix = container.find_element(
                By.CSS_SELECTOR, 'p.price_color'
            ).text

            disponibilite = container.find_element(
                By.CSS_SELECTOR, 'p.instock.availability'
            ).text

            note = container.find_element(
                By.CSS_SELECTOR, 'p.star-rating'
            ).get_attribute('class')

            # Récupérer le lien vers la fiche détaillée
            lien = container.find_element(
                By.CSS_SELECTOR, 'h3 a'
            ).get_attribute('href')

            # Mémoriser les informations avant de quitter la page catalogue
            livres.append({
                'Titre': titre,
                'Prix': prix,
                'Disponibilite': disponibilite,
                'Note': note,
                'Lien': lien
            })

        except:
            pass
# Parcourir les livres mémorisés
    for livre in livres:

        try:
            # Ouvrir la fiche détaillée du livre
            driver.get(livre['Lien'])

            # Extraire le nombre de reviews
            reviews = driver.find_element(
                By.XPATH,
                "//th[text()='Number of reviews']/following-sibling::td"
            ).text

            # Extraire la description
            description = driver.find_element(
                By.XPATH,
                "//div[@id='product_description']/following-sibling::p[1]"
            ).text

            # Extraire le type de produit
            type_produit = driver.find_element(
                By.XPATH,
                "//th[text()='Product Type']/following-sibling::td"
            ).text

            # Extraire la taxe
            tax = driver.find_element(
                By.XPATH,
                "//th[text()='Tax']/following-sibling::td"
            ).text

            # Regrouper toutes les variables du livre
            dic = {
                'Titre': livre['Titre'],
                'Prix': livre['Prix'],
                'Disponibilite': livre['Disponibilite'],
                'Nombre_produits': nombre_produits,
                'Note': livre['Note'],
                'Nombre_reviews': reviews,
                'Description': description,
                'Type_produit': type_produit,
                'Tax': tax
            }

            # Ajouter le livre à la liste des données
            data.append(dic)

        except:
            pass

    # Transformer les données de la page en DataFrame
    df = pd.DataFrame(data)

    # Ajouter les données de la page au DataFrame final
    df_final = pd.concat(
        [df_final, df],
        axis=0
    ).reset_index(drop=True)

df_final.head()
df_final.shape
# Afficher les noms des colonnes du DataFrame
df_final.columns


Index(['Titre', 'Prix', 'Disponibilite', 'Nombre_produits', 'Note',
       'Nombre_reviews', 'Description', 'Type_produit', 'Tax'],
      dtype='object')

# Scraping complet après correction des descriptions manquantes

In [11]:
# Créer un DataFrame final vide
df_final = pd.DataFrame()

# Parcourir toutes les pages du catalogue
for i in range(1, 51):

    # Construire l'URL de chaque page
    url = f'https://books.toscrape.com/catalogue/page-{i}.html'

    # Ouvrir la page
    driver.get(url)

    # Identifier les conteneurs des livres
    containers = driver.find_elements(
        By.CSS_SELECTOR,
        'article.product_pod'
    )

    # Compter le nombre de produits sur la page
    nombre_produits = len(containers)

    # Créer une liste vide pour stocker les données de la page
    data = []

    # Liste temporaire pour mémoriser les informations des livres
    livres = []

    # Parcourir les conteneurs de la page catalogue
    for container in containers:

        try:
            titre = container.find_element(
                By.CSS_SELECTOR,
                'h3 a'
            ).get_attribute('title')

            prix = container.find_element(
                By.CSS_SELECTOR,
                'p.price_color'
            ).text

            disponibilite = container.find_element(
                By.CSS_SELECTOR,
                'p.instock.availability'
            ).text

            note = container.find_element(
                By.CSS_SELECTOR,
                'p.star-rating'
            ).get_attribute('class')

            lien = container.find_element(
                By.CSS_SELECTOR,
                'h3 a'
            ).get_attribute('href')

            livres.append({
                'Titre': titre,
                'Prix': prix,
                'Disponibilite': disponibilite,
                'Note': note,
                'Lien': lien
            })

        except:
            pass

    # Parcourir les livres mémorisés
    for livre in livres:

        try:
            # Ouvrir la fiche détaillée du livre
            driver.get(livre['Lien'])

            # Extraire le nombre de reviews
            reviews = driver.find_element(
                By.XPATH,
                "//th[text()='Number of reviews']/following-sibling::td"
            ).text

            # Extraire la description si elle existe
            try:
                description = driver.find_element(
                    By.XPATH,
                    "//div[@id='product_description']/following-sibling::p[1]"
                ).text
            except:
                description = None

            # Extraire le type de produit
            type_produit = driver.find_element(
                By.XPATH,
                "//th[text()='Product Type']/following-sibling::td"
            ).text

            # Extraire la taxe
            tax = driver.find_element(
                By.XPATH,
                "//th[text()='Tax']/following-sibling::td"
            ).text

            # Regrouper toutes les variables
            dic = {
                'Titre': livre['Titre'],
                'Prix': livre['Prix'],
                'Disponibilite': livre['Disponibilite'],
                'Nombre_produits': nombre_produits,
                'Note': livre['Note'],
                'Nombre_reviews': reviews,
                'Description': description,
                'Type_produit': type_produit,
                'Tax': tax
            }

            # Ajouter le livre à la liste des données
            data.append(dic)

        except:
            pass

    # Transformer les données de la page en DataFrame
    df = pd.DataFrame(data)

    # Ajouter les données au DataFrame final
    df_final = pd.concat(
        [df_final, df],
        axis=0
    ).reset_index(drop=True)

In [12]:
df_final.head()

,Titre,Prix,Disponibilite,Nombre_produits,Note,Nombre_reviews,Description,Type_produit,Tax
0,A Light in the Attic,£51.77,In stock,20,star-rating Three,0,It's hard to imagine a world without A Light i...,Books,£0.00
1,Tipping the Velvet,£53.74,In stock,20,star-rating One,0,"""Erotic and absorbing...Written with starling ...",Books,£0.00
2,Soumission,£50.10,In stock,20,star-rating One,0,"Dans une France assez proche de la nôtre, un h...",Books,£0.00
3,Sharp Objects,£47.82,In stock,20,star-rating Four,0,"WICKED above her hipbone, GIRL across her hear...",Books,£0.00
4,Sapiens: A Brief History of Humankind,£54.23,In stock,20,star-rating Five,0,From a renowned historian comes a groundbreaki...,Books,£0.00


In [13]:
df_final.shape

(1000, 9)

In [14]:
df_final.columns

Index(['Titre', 'Prix', 'Disponibilite', 'Nombre_produits', 'Note',
       'Nombre_reviews', 'Description', 'Type_produit', 'Tax'],
      dtype='object')

In [15]:
df_final.isnull().sum()

,0
Titre,0
Prix,0
Disponibilite,0
Nombre_produits,0
Note,0
Nombre_reviews,0
Description,2
Type_produit,0
Tax,0


Interprétation : deux livres ne disposent pas de description dans leur fiche détaillée. Ces valeurs ont été conservées comme manquantes (None) afin de ne pas supprimer les observations concernées.

**Objectif du nettoyage** : transformer les données brutes collectées en données propres, cohérentes et exploitables pour l’analyse, la base SQL et le dashboard Streamlit.

In [16]:
df_final.dtypes

,0
Titre,object
Prix,object
Disponibilite,object
Nombre_produits,int64
Note,object
Nombre_reviews,object
Description,object
Type_produit,object
Tax,object


In [17]:
df_final['Note'].unique()

array(['star-rating Three', 'star-rating One', 'star-rating Four',
       'star-rating Five', 'star-rating Two'], dtype=object)

# Nettoyage de la variable Note: l'objectif est de passer de
star-rating Three → 3

star-rating One   → 1

star-rating Four  → 4

star-rating Five  → 5

star-rating Two   → 2

In [18]:
# Dictionnaire de correspondance des notes
conversion_note = {
    'star-rating One': 1,
    'star-rating Two': 2,
    'star-rating Three': 3,
    'star-rating Four': 4,
    'star-rating Five': 5
}

# Transformer les notes textuelles en valeurs numériques
df_final['Note'] = df_final['Note'].map(conversion_note)

Vérification

In [19]:
df_final['Note'].unique()

array([3, 1, 4, 5, 2])

Récupération des notes après erreur de transformation

In [20]:
notes = []

for i in range(1, 51):

    url = f'https://books.toscrape.com/catalogue/page-{i}.html'
    driver.get(url)

    containers = driver.find_elements(
        By.CSS_SELECTOR,
        'article.product_pod'
    )

    for container in containers:
        note = container.find_element(
            By.CSS_SELECTOR,
            'p.star-rating'
        ).get_attribute('class')

        notes.append(note)

In [21]:
len(notes)

1000

In [22]:
notes[:5]

['star-rating Three',
 'star-rating One',
 'star-rating One',
 'star-rating Four',
 'star-rating Five']

In [23]:
df_final['Note'] = notes

In [24]:
conversion_note = {
    'star-rating One': 1,
    'star-rating Two': 2,
    'star-rating Three': 3,
    'star-rating Four': 4,
    'star-rating Five': 5
}

df_final['Note'] = df_final['Note'].map(conversion_note)

In [25]:
df_final['Note'].unique()

array([3, 1, 4, 5, 2])

Nombre_reviews

In [26]:
df_final['Nombre_reviews'].unique()

array(['0'], dtype=object)

In [27]:
df_final['Nombre_reviews'].dtype

dtype('O')

In [28]:
df_final['Nombre_reviews'] = df_final['Nombre_reviews'].astype(int)

In [29]:
df_final['Nombre_reviews'].dtype

dtype('int64')

# La variable Tax

In [30]:
df_final['Tax'].unique()

array(['£0.00'], dtype=object)

In [31]:
# Nettoyer la taxe et la convertir en nombre décimal
df_final['Tax'] = (
    df_final['Tax']
    .str.replace('£', '', regex=False)
    .astype(float)
)

In [32]:
df_final['Tax'].unique()

array([0.])

In [33]:
df_final['Tax'].dtype

dtype('float64')

La variable Disponibilité

In [34]:
df_final['Disponibilite'].unique()

array(['In stock'], dtype=object)

La variable Type_Produit

In [35]:
df_final['Type_produit'].unique()

array(['Books'], dtype=object)

La variable Description

In [36]:
df_final[df_final['Description'].isnull()]

,Titre,Prix,Disponibilite,Nombre_produits,Note,Nombre_reviews,Description,Type_produit,Tax
160,The Bridge to Consciousness: I'm Writing the B...,£32.00,In stock,20,3,0,None,Books,0.0
995,Alice in Wonderland (Alice's Adventures in Won...,£55.53,In stock,20,1,0,None,Books,0.0


In [37]:
# Remplacer les descriptions manquantes
df_final['Description'] = df_final['Description'].fillna(
    'Description non disponible'
)

In [38]:
df_final['Description'].isnull().sum()

np.int64(0)

contrôle final de tout le nettoyage

In [39]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Titre            1000 non-null   object 
 1   Prix             1000 non-null   object 
 2   Disponibilite    1000 non-null   object 
 3   Nombre_produits  1000 non-null   int64  
 4   Note             1000 non-null   int64  
 5   Nombre_reviews   1000 non-null   int64  
 6   Description      1000 non-null   object 
 7   Type_produit     1000 non-null   object 
 8   Tax              1000 non-null   float64
dtypes: float64(1), int64(3), object(5)
memory usage: 70.4+ KB


In [40]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Titre            1000 non-null   object 
 1   Prix             1000 non-null   object 
 2   Disponibilite    1000 non-null   object 
 3   Nombre_produits  1000 non-null   int64  
 4   Note             1000 non-null   int64  
 5   Nombre_reviews   1000 non-null   int64  
 6   Description      1000 non-null   object 
 7   Type_produit     1000 non-null   object 
 8   Tax              1000 non-null   float64
dtypes: float64(1), int64(3), object(5)
memory usage: 70.4+ KB


In [41]:
# Vérifier la présence de doublons
df_final.duplicated().sum()

np.int64(0)

**Conclusion du nettoyage des données**
Le nettoyage des données a permis de vérifier et corriger les types de variables, de transformer les notes et les variables numériques, de traiter les deux descriptions manquantes et de contrôler les doublons. Le jeu de données final contient 1000 observations et 9 variables, sans valeur manquante ni doublon, et est désormais prêt pour les étapes suivantes du projet.

**********************************************************************

# **Scraping du site Gaaraas**
Cette étape consiste à explorer la structure HTML du site Gaaraas, identifier les conteneurs et les sélecteurs correspondant aux variables demandées, puis automatiser leur collecte avec Selenium.

# Ouverture de la page Gaaraas avec Selenium

In [42]:
url_gaaraas = 'https://www.gaaraas.com/fr/users/dakar-auto?page=2'

driver.get(url_gaaraas)

In [43]:
# Identifier les conteneurs des voitures
containers_gaaraas = driver.find_elements(
    By.CSS_SELECTOR,
    'a.common-ad-card'
)

len(containers_gaaraas)

20

# Test d’extraction des variables sur une seule annonce

In [44]:
# Sélectionner la première voiture
container_gaaraas = containers_gaaraas[0]

In [45]:
titre_voiture = container_gaaraas.find_element(
    By.CSS_SELECTOR,
    'div.ad-specification h4'
).text

print(titre_voiture)

2005 Renault Scenic


In [46]:
region = container_gaaraas.find_element(
    By.CSS_SELECTOR,
    'div.location'
).text

print(region)

Dakar


In [47]:
prix = container_gaaraas.find_element(
    By.CSS_SELECTOR,
    'div.ad-vehicle-price div.value'
).text

print(prix)

CFA 2 200 000


In [48]:
kilometrage = container_gaaraas.find_element(
    By.CSS_SELECTOR,
    'div.ad-vehicle-mileage div.value'
).text

print(kilometrage)

140 000 KM


In [49]:
boite_vitesse = container_gaaraas.find_element(
    By.CSS_SELECTOR,
    'div.transmission span'
).text

print(boite_vitesse)

Manuelle


In [50]:
# Récupérer le lien vers la fiche détaillée de la voiture
lien_voiture = container_gaaraas.get_attribute('href')

print(lien_voiture)

https://www.gaaraas.com/fr/vehicle_listings/annonce-renault-scenic-dakar-dakar-2610


In [51]:
driver.get(lien_voiture)

In [52]:
print(driver.current_url)

https://www.gaaraas.com/fr/vehicle_listings/annonce-renault-scenic-dakar-dakar-2610


In [53]:
driver.get(url_gaaraas)

containers_gaaraas = driver.find_elements(
    By.CSS_SELECTOR,
    'a.common-ad-card'
)

for container in containers_gaaraas[:10]:
    titre = container.find_element(
        By.CSS_SELECTOR,
        'div.ad-specification h4'
    ).text

    print(titre)

2005 Renault Scenic
2005 Renault Megane
2011 Honda Elysion
2010 Dacia Logan
2007 Honda Civic
2007 Skoda Fabia
2009 Toyota Aygo
2012 Chevrolet Camaro
2000 Peugeot 309
2008 Peugeot 307


# Séparation de l’année, de la marque et du modèle

In [54]:
# Fonction permettant de séparer le titre de la voiture
def separer_voiture(titre):

    parties = titre.split()

    # L'année correspond au premier élément
    annee = parties[0]

    # Cas des marques composées
    if parties[1] == 'Land' and parties[2] == 'Rover':
        marque = 'Land Rover'
        modele = ' '.join(parties[3:])

    # Cas général : marque composée d'un seul mot
    else:
        marque = parties[1]
        modele = ' '.join(parties[2:])

    return annee, marque, modele

In [55]:
annee, marque, modele = separer_voiture(titre_voiture)

print("Année :", annee)
print("Marque :", marque)
print("Modèle :", modele)

Année : 2005
Marque : Renault
Modèle : Scenic


# Creéation de notre premier dictionnaire Gaaraas

In [56]:
dic_gaaraas = {
    'Marque': marque,
    'Modele': modele,
    'Annee': annee,
    'Prix': prix,
    'Kilometrage': kilometrage,
    'Boite_vitesse': boite_vitesse,
    'Region': region
}

dic_gaaraas

{'Marque': 'Renault',
 'Modele': 'Scenic',
 'Annee': '2005',
 'Prix': 'CFA 2 200 000',
 'Kilometrage': '140 000 KM',
 'Boite_vitesse': 'Manuelle',
 'Region': 'Dakar'}

Test sur UNE voiture validé à 100 %.

In [57]:
# Revenir à la page des annonces
driver.get(url_gaaraas)

# Recréer les conteneurs
containers_gaaraas = driver.find_elements(
    By.CSS_SELECTOR,
    'a.common-ad-card'
)

len(containers_gaaraas)

20

# Test de collecte sur une page

In [58]:
# Créer une liste vide pour stocker les voitures
data_gaaraas = []

# Parcourir les 20 voitures de la page
for container in containers_gaaraas:

    try:
        # Récupérer le titre complet
        titre_voiture = container.find_element(
            By.CSS_SELECTOR,
            'div.ad-specification h4'
        ).text

        # Séparer année, marque et modèle
        annee, marque, modele = separer_voiture(titre_voiture)

        # Récupérer la région
        region = container.find_element(
            By.CSS_SELECTOR,
            'div.location'
        ).text

        # Récupérer le prix
        prix = container.find_element(
            By.CSS_SELECTOR,
            'div.ad-vehicle-price div.value'
        ).text

        # Récupérer le kilométrage
        kilometrage = container.find_element(
            By.CSS_SELECTOR,
            'div.ad-vehicle-mileage div.value'
        ).text

        # Récupérer la boîte de vitesses
        boite_vitesse = container.find_element(
            By.CSS_SELECTOR,
            'div.transmission span'
        ).text

        # Regrouper les informations de la voiture
        dic_gaaraas = {
            'Marque': marque,
            'Modele': modele,
            'Annee': annee,
            'Prix': prix,
            'Kilometrage': kilometrage,
            'Boite_vitesse': boite_vitesse,
            'Region': region
        }

        # Ajouter la voiture à la liste
        data_gaaraas.append(dic_gaaraas)

    except:
        pass

In [59]:
df_gaaraas = pd.DataFrame(data_gaaraas)

In [60]:
df_gaaraas.head()

,Marque,Modele,Annee,Prix,Kilometrage,Boite_vitesse,Region
0,Renault,Scenic,2005,CFA 2 200 000,140 000 KM,Manuelle,Dakar
1,Renault,Megane,2005,CFA 2 000 000,160 000 KM,Manuelle,Dakar
2,Honda,Elysion,2011,CFA 3 500 000,160 000 KM,Automatique,Dakar
3,Dacia,Logan,2010,CFA 2 000 000,161 000 KM,Manuelle,Dakar
4,Honda,Civic,2007,CFA 2 300 000,180 000 KM,Automatique,Dakar


In [61]:
df_gaaraas.shape

(20, 7)

In [62]:
df_gaaraas.columns

Index(['Marque', 'Modele', 'Annee', 'Prix', 'Kilometrage', 'Boite_vitesse',
       'Region'],
      dtype='object')

**Validation du test sur une page :** le scraper retourne 20 observations et 7 variables, ce qui correspond aux 20 annonces détectées sur la page et aux 7 variables demandées pour Gaaraas. Ce test permet de valider les sélecteurs avant d’étendre le scraping à plusieurs pages.

# Test sur 2 pages avec une boucle

# Test du scraping Gaaraas sur deux pages

In [63]:
# Créer un DataFrame final vide pour Gaaraas
df_gaaraas_final = pd.DataFrame()

# Parcourir les pages 1 et 2
for i in range(1, 3):

    # Construire l'URL de chaque page
    url = f'https://www.gaaraas.com/fr/users/dakar-auto?page={i}'

    # Ouvrir la page
    driver.get(url)

    # Identifier les conteneurs des voitures
    containers_gaaraas = driver.find_elements(
        By.CSS_SELECTOR,
        'a.common-ad-card'
    )

    # Créer une liste vide pour stocker les données de la page
    data_gaaraas = []

    # Parcourir les voitures de la page
    for container in containers_gaaraas:

        try:
            # Récupérer le titre complet
            titre_voiture = container.find_element(
                By.CSS_SELECTOR,
                'div.ad-specification h4'
            ).text

            # Séparer année, marque et modèle
            annee, marque, modele = separer_voiture(titre_voiture)

            # Récupérer la région
            region = container.find_element(
                By.CSS_SELECTOR,
                'div.location'
            ).text

            # Récupérer le prix
            prix = container.find_element(
                By.CSS_SELECTOR,
                'div.ad-vehicle-price div.value'
            ).text

            # Récupérer le kilométrage
            kilometrage = container.find_element(
                By.CSS_SELECTOR,
                'div.ad-vehicle-mileage div.value'
            ).text

            # Récupérer la boîte de vitesses
            boite_vitesse = container.find_element(
                By.CSS_SELECTOR,
                'div.transmission span'
            ).text

            # Regrouper les informations
            dic_gaaraas = {
                'Marque': marque,
                'Modele': modele,
                'Annee': annee,
                'Prix': prix,
                'Kilometrage': kilometrage,
                'Boite_vitesse': boite_vitesse,
                'Region': region
            }

            # Ajouter l'observation
            data_gaaraas.append(dic_gaaraas)

        except:
            pass

    # Transformer les données de la page en DataFrame
    df = pd.DataFrame(data_gaaraas)

    # Ajouter les données au DataFrame final
    df_gaaraas_final = pd.concat(
        [df_gaaraas_final, df],
        axis=0
    ).reset_index(drop=True)

In [64]:
df_gaaraas_final.head()

,Marque,Modele,Annee,Prix,Kilometrage,Boite_vitesse,Region
0,Nissan,Murano,2008,CFA 3 200 000,115 305 KM,Automatique,Dakar
1,Ford,Escort,2007,CFA 2 700 000,174 000 KM,Manuelle,Dakar
2,Nissan,Qashqai,2014,CFA 7 800 000,63 000 KM,Automatique,Dakar
3,Mazda,6,2010,CFA 4 000 000,182 000 KM,Automatique,Dakar
4,Peugeot,307,2007,CFA 2 400 000,160 000 KM,Manuelle,Dakar


In [65]:
df_gaaraas_final.shape

(40, 7)

### Amélioration de la gestion des erreurs

Lors des premiers tests, les erreurs étaient gérées avec `except: pass`. Cette méthode permet au programme de continuer son exécution lorsqu'une erreur est rencontrée, mais elle masque complètement l'origine du problème.

L'expérience du scraping de ***Books to Scrape*** a montré qu'une donnée manquante pouvait ainsi entraîner la perte silencieuse de certaines observations.

Pour le scraping complet de Gaaraas, ***except pass*** est donc remplacé par une gestion plus explicite des erreurs :
***except Exception as e***

Cette modification permet au programme de continuer le scraping tout en affichant la page concernée et le message d'erreur. Il devient ainsi possible d'identifier et d'analyser les annonces qui n'ont pas pu être collectées.

# Lancement du sraping sur les 100 pages

In [66]:
# Créer un DataFrame final vide pour Gaaraas
df_gaaraas_final = pd.DataFrame()

# Parcourir les 100 pages
for i in range(1, 101):

    # Construire l'URL de chaque page
    url = f'https://www.gaaraas.com/fr/users/dakar-auto?page={i}'

    # Ouvrir la page
    driver.get(url)

    # Identifier les conteneurs des voitures
    containers_gaaraas = driver.find_elements(
        By.CSS_SELECTOR,
        'a.common-ad-card'
    )

    # Créer une liste vide pour stocker les données de la page
    data_gaaraas = []

    # Parcourir les voitures de la page
    for container in containers_gaaraas:

        try:
            # Récupérer le titre complet
            titre_voiture = container.find_element(
                By.CSS_SELECTOR,
                'div.ad-specification h4'
            ).text

            # Séparer année, marque et modèle
            annee, marque, modele = separer_voiture(titre_voiture)

            # Récupérer la région
            region = container.find_element(
                By.CSS_SELECTOR,
                'div.location'
            ).text

            # Récupérer le prix
            prix = container.find_element(
                By.CSS_SELECTOR,
                'div.ad-vehicle-price div.value'
            ).text

            # Récupérer le kilométrage
            kilometrage = container.find_element(
                By.CSS_SELECTOR,
                'div.ad-vehicle-mileage div.value'
            ).text

            # Récupérer la boîte de vitesses
            boite_vitesse = container.find_element(
                By.CSS_SELECTOR,
                'div.transmission span'
            ).text

            # Regrouper les informations
            dic_gaaraas = {
                'Marque': marque,
                'Modele': modele,
                'Annee': annee,
                'Prix': prix,
                'Kilometrage': kilometrage,
                'Boite_vitesse': boite_vitesse,
                'Region': region
            }

            # Ajouter l'observation
            data_gaaraas.append(dic_gaaraas)

        except Exception as e:
            print(f"Erreur page {i} - annonce ignorée")
            print(e)

    # Transformer les données de la page en DataFrame
    df = pd.DataFrame(data_gaaraas)

    # Ajouter les données au DataFrame final
    df_gaaraas_final = pd.concat(
        [df_gaaraas_final, df],
        axis=0
    ).reset_index(drop=True)

    # Afficher la progression
    print(f"Page {i} terminée")

Page 1 terminée
Page 2 terminée
Page 3 terminée
Page 4 terminée
Page 5 terminée
Page 6 terminée
Page 7 terminée
Page 8 terminée
Erreur page 9 - annonce ignorée
Message: no such element: Unable to locate element: {"method":"css selector","selector":"div.ad-vehicle-mileage div.value"}
  (Session info: chrome=151.0.7922.137); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
#0 0x5d50243e136a <unknown>
#1 0x5d5023d54f49 <unknown>
#2 0x5d5023daa7e2 <unknown>
#3 0x5d5023daaa31 <unknown>
#4 0x5d5023d9f6a6 <unknown>
#5 0x5d5023d9f587 <unknown>
#6 0x5d5023df30c3 <unknown>
#7 0x5d5023d9dc92 <unknown>
#8 0x5d5023d9eb11 <unknown>
#9 0x5d50243a68d0 <unknown>
#10 0x5d50243a4f3a <unknown>
#11 0x5d502438f9b5 <unknown>
#12 0x5d50243a5c0a <unknown>
#13 0x5d5024377740 <unknown>
#14 0x5d50243cc9a8 <unknown>
#15 0x5d50243ccb45 <unknown>
#16 0x5d50243dff1e <unknown>
#17 0x7c614a76ba83 <unknown>

Erreur p

In [67]:
df_gaaraas_final.shape

(243, 7)

Diagnostic du nombre d’annonces par page

In [68]:
# Vérifier le nombre de conteneurs trouvés sur chaque page
for i in range(1, 101):

    url = f'https://www.gaaraas.com/fr/users/dakar-auto?page={i}'
    driver.get(url)

    containers = driver.find_elements(
        By.CSS_SELECTOR,
        'a.common-ad-card'
    )

    print(f"Page {i} : {len(containers)} annonces")

Page 1 : 20 annonces
Page 2 : 20 annonces
Page 3 : 20 annonces
Page 4 : 20 annonces
Page 5 : 20 annonces
Page 6 : 20 annonces
Page 7 : 20 annonces
Page 8 : 20 annonces
Page 9 : 20 annonces
Page 10 : 20 annonces
Page 11 : 20 annonces
Page 12 : 20 annonces
Page 13 : 5 annonces
Page 14 : 0 annonces
Page 15 : 0 annonces
Page 16 : 0 annonces
Page 17 : 0 annonces
Page 18 : 0 annonces
Page 19 : 0 annonces
Page 20 : 0 annonces
Page 21 : 0 annonces
Page 22 : 0 annonces
Page 23 : 0 annonces
Page 24 : 0 annonces
Page 25 : 0 annonces
Page 26 : 0 annonces
Page 27 : 0 annonces
Page 28 : 0 annonces
Page 29 : 0 annonces
Page 30 : 0 annonces
Page 31 : 0 annonces
Page 32 : 0 annonces
Page 33 : 0 annonces
Page 34 : 0 annonces
Page 35 : 0 annonces
Page 36 : 0 annonces
Page 37 : 0 annonces
Page 38 : 0 annonces
Page 39 : 0 annonces
Page 40 : 0 annonces
Page 41 : 0 annonces
Page 42 : 0 annonces
Page 43 : 0 annonces
Page 44 : 0 annonces
Page 45 : 0 annonces
Page 46 : 0 annonces
Page 47 : 0 annonces
Page 48 : 

**Observation sur la pagination** : le sujet demande de parcourir 100 pages du site Gaaraas. La boucle a donc été construite sur les pages 1 à 100 conformément à cette consigne. Lors de l’exécution, l’observation du site montre cependant que les annonces disponibles s’arrêtent actuellement autour de la page 13. Les pages suivantes ne retournent donc aucun conteneur correspondant aux annonces. Cette limitation provient de l’état actuel de la source de données et non du processus de scraping.

In [69]:
# Créer un DataFrame final vide pour Gaaraas
df_gaaraas_final = pd.DataFrame()

# Parcourir les 100 pages demandées
for i in range(1, 101):

    # Construire l'URL de chaque page
    url = f'https://www.gaaraas.com/fr/users/dakar-auto?page={i}'

    # Ouvrir la page
    driver.get(url)

    # Identifier les conteneurs des voitures
    containers_gaaraas = driver.find_elements(
        By.CSS_SELECTOR,
        'a.common-ad-card'
    )

    # Créer une liste vide pour stocker les données de la page
    data_gaaraas = []

    # Parcourir les voitures de la page
    for container in containers_gaaraas:

        try:
            # Récupérer le titre complet
            titre_voiture = container.find_element(
                By.CSS_SELECTOR,
                'div.ad-specification h4'
            ).text

            # Séparer année, marque et modèle
            annee, marque, modele = separer_voiture(titre_voiture)

            # Récupérer la région
            region = container.find_element(
                By.CSS_SELECTOR,
                'div.location'
            ).text

            # Récupérer le prix
            prix = container.find_element(
                By.CSS_SELECTOR,
                'div.ad-vehicle-price div.value'
            ).text

            # Récupérer le kilométrage
            # Si le kilométrage est absent, conserver la voiture
            try:
                kilometrage = container.find_element(
                    By.CSS_SELECTOR,
                    'div.ad-vehicle-mileage div.value'
                ).text
            except:
                kilometrage = None

            # Récupérer la boîte de vitesses
            boite_vitesse = container.find_element(
                By.CSS_SELECTOR,
                'div.transmission span'
            ).text

            # Regrouper les informations
            dic_gaaraas = {
                'Marque': marque,
                'Modele': modele,
                'Annee': annee,
                'Prix': prix,
                'Kilometrage': kilometrage,
                'Boite_vitesse': boite_vitesse,
                'Region': region
            }

            # Ajouter la voiture à la liste
            data_gaaraas.append(dic_gaaraas)

        # Afficher les autres erreurs éventuelles
        except Exception as e:
            print(f"Erreur page {i} - annonce ignorée")
            print(e)

    # Transformer les données de la page en DataFrame
    df = pd.DataFrame(data_gaaraas)

    # Ajouter les données au DataFrame final
    df_gaaraas_final = pd.concat(
        [df_gaaraas_final, df],
        axis=0
    ).reset_index(drop=True)

    # Afficher la progression
    print(f"Page {i} terminée")

Page 1 terminée
Page 2 terminée
Page 3 terminée
Page 4 terminée
Page 5 terminée
Page 6 terminée
Page 7 terminée
Page 8 terminée
Page 9 terminée
Page 10 terminée
Page 11 terminée
Page 12 terminée
Page 13 terminée
Page 14 terminée
Page 15 terminée
Page 16 terminée
Page 17 terminée
Page 18 terminée
Page 19 terminée
Page 20 terminée
Page 21 terminée
Page 22 terminée
Page 23 terminée
Page 24 terminée
Page 25 terminée
Page 26 terminée
Page 27 terminée
Page 28 terminée
Page 29 terminée
Page 30 terminée
Page 31 terminée
Page 32 terminée
Page 33 terminée
Page 34 terminée
Page 35 terminée
Page 36 terminée
Page 37 terminée
Page 38 terminée
Page 39 terminée
Page 40 terminée
Page 41 terminée
Page 42 terminée
Page 43 terminée
Page 44 terminée
Page 45 terminée
Page 46 terminée
Page 47 terminée
Page 48 terminée
Page 49 terminée
Page 50 terminée
Page 51 terminée
Page 52 terminée
Page 53 terminée
Page 54 terminée
Page 55 terminée
Page 56 terminée
Page 57 terminée
Page 58 terminée
Page 59 terminée
Page 6

In [70]:
df_gaaraas_final.shape

(245, 7)

In [71]:
df_gaaraas_final.isnull().sum()

,0
Marque,0
Modele,0
Annee,0
Prix,0
Kilometrage,2
Boite_vitesse,0
Region,0


In [72]:
df_gaaraas_final.dtypes

,0
Marque,object
Modele,object
Annee,object
Prix,object
Kilometrage,object
Boite_vitesse,object
Region,object


In [73]:
df_gaaraas_final['Annee'].unique()

array(['2008', '2007', '2014', '2010', '2003', '2011', 'Citroen', '2005',
       '2006', '2004', '2012', '2009', '2000', '1989', '2002', '2013',
       '2015'], dtype=object)

### Détection d'une anomalie dans la variable "Annee"

L'exploration des valeurs uniques de la colonne "Annee" révèle la présence de la valeur "**Citroen**", alors que cette variable devrait contenir uniquement des années.

Cette anomalie indique qu'au moins une annonce n'a pas été correctement décomposée lors de la séparation du titre en "Annee", "MArque" et "Modele".

Plutôt que de remplacer directement cette valeur, l'observation concernée sera d'abord identifiée afin de comprendre l'origine de l'anomalie et, si nécessaire, d'améliorer la fonction "separer_voiture".

Cette vérification illustre l'importance d'explorer les valeurs d'une variable avant d'effectuer une conversion de type.

In [74]:
df_gaaraas_final[
    df_gaaraas_final['Annee'] == 'Citroen'
]

,Marque,Modele,Annee,Prix,Kilometrage,Boite_vitesse,Region
10,C3,,Citroen,CFA 1 800 000,220 000 KM,Manuelle,Dakar


### Identification de l'observation problématique

L'observation contenant "Citroen" dans la variable "Annee" a été identifiée. Elle présente également une inversion apparente des informations : "C3" apparaît dans  "Marque", tandis que  "Citroen" apparaît dans  "Annee".

Cette observation montre que la structure de son titre diffère probablement du format général  "Année +Marque+Mdèle" utilisé pour construire la fonction "separer_voiture"().

Avant toute correction, le titre original de cette annonce doit être examiné afin d'identifier précisément la cause de cette mauvaise séparation.

In [75]:
# Retourner sur la première page
driver.get('https://www.gaaraas.com/fr/users/dakar-auto?page=1')

# Récupérer les conteneurs
containers_gaaraas = driver.find_elements(
    By.CSS_SELECTOR,
    'a.common-ad-card'
)

# Afficher les titres avec leur position
for i, container in enumerate(containers_gaaraas):
    titre = container.find_element(
        By.CSS_SELECTOR,
        'div.ad-specification h4'
    ).text

    print(i, titre)

0 2008 Nissan Murano
1 2007 Ford Escort
2 2014 Nissan Qashqai
3 2010 Mazda 6
4 2007 Peugeot 307
5 2008 Citroen C3
6 2003 Volkswagen Golf
7 2008 Hyundai Tucson
8 2011 Nissan Murano
9 2008 Citroen Berlingo
10 Citroen C3
11 2011 Ford Edge
12 2008 Land Rover Range Rover Sport
13 2005 Toyota Auris
14 2006 Mitsubishi L200
15 2004 Mercedes‒Benz 190
16 2012 Peugeot 308
17 2009 Renault Scenic
18 2014 Nissan Altima
19 2004 Peugeot 605


### Origine de l'anomalie détectée dans "Annee"

L'examen du titre original de l'observation problématique montre que l'annonce est intitulée  "Citroen C3", contrairement aux autres annonces qui suivent généralement la structure  "Année +Marque+Mdèle" .

L'année n'est donc pas renseignée dans cette annonce. La fonction "separer_voiture"() considérait cependant systématiquement le premier élément du titre comme une année, ce qui a entraîné un décalage des informations.

La bonne interprétation de cette annonce est donc :

- Annee: valeur manquante
- Marque: Citroen
- Modele: C3

La fonction de séparation doit ainsi être améliorée afin de vérifier si le premier élément du titre correspond réellement à une année avant d'effectuer la séparation.

### Amélioration de la fonction separer_voiture

L'analyse de l'annonce Citroen C3 montre que l'année n'est pas toujours renseignée dans le titre des véhicules. La première fonction créée considérait pourtant que le premier élément du titre était toujours l'année.

La fonction est donc améliorée pour vérifier d'abord si le premier élément est numérique. S'il est numérique, il est considéré comme l'année. Dans le cas contraire, l'année est considérée comme une valeur manquante et la séparation de la marque et du modèle se poursuit normalement.

# Le cas particulier des marques composées comme Land Rover est également conservé.

In [76]:
# Amélioration de la fonction de séparation
def separer_voiture(titre):

    parties = titre.split()

    # Vérifier si le premier élément correspond à une année
    if parties[0].isdigit():
        annee = parties[0]
        debut_marque = 1

    # Si l'année est absente
    else:
        annee = None
        debut_marque = 0

    # Cas particulier : marque composée Land Rover
    if (
        len(parties) > debut_marque + 1
        and parties[debut_marque] == 'Land'
        and parties[debut_marque + 1] == 'Rover'
    ):
        marque = 'Land Rover'
        modele = ' '.join(parties[debut_marque + 2:])

    # Cas général
    else:
        marque = parties[debut_marque]
        modele = ' '.join(parties[debut_marque + 1:])

    return annee, marque, modele

In [77]:
separer_voiture('2008 Land Rover Range Rover Sport')

('2008', 'Land Rover', 'Range Rover Sport')

In [78]:
separer_voiture('Citroen C3')

(None, 'Citroen', 'C3')

### Nouvelle collecte après amélioration de la fonction

Les deux tests montrent que la nouvelle fonction sépare correctement les titres avec une année et les titres sans année. La collecte est donc relancée avec cette fonction améliorée afin d'obtenir le DataFrame final corrigé.

In [79]:
df_gaaraas_final.shape

(245, 7)

In [80]:
df_gaaraas_final['Annee'].unique()

array(['2008', '2007', '2014', '2010', '2003', '2011', 'Citroen', '2005',
       '2006', '2004', '2012', '2009', '2000', '1989', '2002', '2013',
       '2015'], dtype=object)

In [81]:
df_gaaraas_final.isnull().sum()

,0
Marque,0
Modele,0
Annee,0
Prix,0
Kilometrage,2
Boite_vitesse,0
Region,0


In [82]:
df_gaaraas_final['Annee'] = pd.to_numeric(
    df_gaaraas_final['Annee'],
    errors='coerce'
).astype('Int64')

In [83]:
df_gaaraas_final['Annee'].dtype

Int64Dtype()

In [84]:
df_gaaraas_final['Annee'].unique()

<IntegerArray>
[2008, 2007, 2014, 2010, 2003, 2011, <NA>, 2005, 2006, 2004, 2012, 2009, 2000,
 1989, 2002, 2013, 2015]
Length: 17, dtype: Int64

In [85]:
df_gaaraas_final['Prix'].unique()

array(['CFA 3 200 000', 'CFA 2 700 000', 'CFA 7 800 000', 'CFA 4 000 000',
       'CFA 2 400 000', 'CFA 2 300 000', 'CFA 2 500 000', 'CFA 3 400 000',
       'CFA 3 300 000', 'CFA 1 800 000', 'CFA 9 800 000', 'CFA 6 000 000',
       'CFA 3 500 000', 'CFA 7 500 000', 'CFA 2 200 000', 'CFA 3 800 000',
       'CFA 1 300 000', 'CFA 2 000 000', 'CFA 2 800 000', 'CFA 2 600 000',
       'CFA 1 900 000', 'CFA 3 000 000', 'CFA 2 900 000', 'CFA 2 150 000',
       'CFA 1 700 000', 'CFA 7 200 000', 'CFA 2 100 000', 'CFA 7 000 000',
       'CFA 11 900 000', 'CFA 1 500 000', 'CFA 3 600 000',
       'CFA 4 500 000', 'CFA 1 600 000', 'CFA 6 500 000', 'CFA 3 900 000',
       'CFA 3 700 000', 'Négociable', 'CFA 5 500 000', 'CFA 10 000 000',
       'CFA 4 300 000', 'CFA 5 800 000', 'CFA 1 950 000', 'CFA 1 400 000',
       'CFA 1 100 000', 'CFA 4 200 000', 'CFA 5 200 000', 'CFA 1 750 000',
       'CFA 4 700 000', 'CFA 8 500 000', 'CFA 4 800 000', 'CFA 8 000 000',
       'CFA 9 500 000', 'CFA 20 000 000', '

In [86]:
df_gaaraas_final[
    df_gaaraas_final['Prix'] == 'Négociable'
]

,Marque,Modele,Annee,Prix,Kilometrage,Boite_vitesse,Region
122,Land Rover,Range Rover Sport,2009,Négociable,160 000 KM,Automatique,Dakar


### Cas particulier dans la variable Prix

L'exploration des valeurs de la variable Prix montre qu'une annonce contient la valeur Négociable à la place d'un montant.

Cette valeur ne peut pas être convertie en nombre. Elle sera donc considérée comme une valeur manquante afin de conserver l'annonce tout en permettant la conversion de la variable Prix en type numérique.

In [87]:
# Nettoyer la variable Prix
df_gaaraas_final['Prix'] = (
    df_gaaraas_final['Prix']
    .str.replace('CFA', '', regex=False)
    .str.replace(' ', '', regex=False)
)

# Convertir en numérique
# La valeur Négociable deviendra automatiquement une valeur manquante
df_gaaraas_final['Prix'] = pd.to_numeric(
    df_gaaraas_final['Prix'],
    errors='coerce'
)

In [88]:
df_gaaraas_final['Prix'].dtype

dtype('float64')

In [89]:
df_gaaraas_final['Prix'].isnull().sum()

np.int64(1)

In [90]:
df_gaaraas_final['Kilometrage'].unique()

array(['115 305 KM', '174 000 KM', '63 000 KM', '182 000 KM',
       '160 000 KM', '128 000 KM', '140 000 KM', '188 000 KM',
       '214 000 KM', '220 000 KM', '80 000 KM', '130 000 KM',
       '179 315 KM', '103 000 KM', '110 000 KM', '161 000 KM',
       '180 000 KM', '150 000 KM', '250 000 KM', '0 KM', '70 000 KM',
       '66 000 KM', '164 231 KM', '164 686 KM', '200 000 KM',
       '265 000 KM', '98 000 KM', '120 000 KM', '84 000 KM', '18 000 KM',
       '311 368 KM', '109 400 KM', '210 000 KM', '125 000 KM',
       '116 000 KM', '172 900 KM', '256 730 KM', '334 000 KM',
       '145 000 KM', '90 000 KM', '115 000 KM', '142 000 KM',
       '230 000 KM', '164 000 KM', '94 954 KM', '132 000 KM',
       '136 000 KM', '28 752 KM', '144 000 KM', '105 000 KM',
       '147 000 KM', '144 410 KM', '137 000 KM', '101 000 KM',
       '21 000 KM', '100 000 KM', '179 000 KM', '291 000 KM',
       '264 127 KM', None, '289 983 KM', '79 044 KM', '99 000 KM',
       '92 000 KM', '88 997 KM', '235 00

In [91]:
# Nettoyer la variable Kilometrage
df_gaaraas_final['Kilometrage'] = (
    df_gaaraas_final['Kilometrage']
    .str.replace('KM', '', regex=False)
    .str.replace(' ', '', regex=False)
)

# Convertir en numérique
df_gaaraas_final['Kilometrage'] = pd.to_numeric(
    df_gaaraas_final['Kilometrage'],
    errors='coerce'
)

In [92]:
df_gaaraas_final['Kilometrage'].dtype

dtype('float64')

In [93]:
df_gaaraas_final['Kilometrage'].isnull().sum()

np.int64(2)

In [94]:
df_gaaraas_final['Boite_vitesse'].value_counts(dropna=False)

,count
Boite_vitesse,
Manuelle,200
Automatique,45


In [95]:
df_gaaraas_final['Region'].value_counts(dropna=False)

,count
Region,
Dakar,244
Dahra,1


In [96]:
df_gaaraas_final[
    df_gaaraas_final['Region'] == 'Dahra'
]

,Marque,Modele,Annee,Prix,Kilometrage,Boite_vitesse,Region
178,Hyundai,Sonata,2013,5200000.0,92000.0,Automatique,Dahra


In [97]:
df_gaaraas_final['Marque'].value_counts()

,count
Marque,
Peugeot,65
Renault,34
Citroen,23
Toyota,21
Ford,17
Hyundai,14
Nissan,12
Mitsubishi,8
Volkswagen,7


In [98]:
# Vérifier les modèles manquants
df_gaaraas_final['Modele'].isnull().sum()

np.int64(0)

In [99]:
# Vérifier les modèles vides
(df_gaaraas_final['Modele'].str.strip() == '').sum()

np.int64(1)

In [100]:
df_gaaraas_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Marque         245 non-null    object 
 1   Modele         245 non-null    object 
 2   Annee          244 non-null    Int64  
 3   Prix           244 non-null    float64
 4   Kilometrage    243 non-null    float64
 5   Boite_vitesse  245 non-null    object 
 6   Region         245 non-null    object 
dtypes: Int64(1), float64(2), object(4)
memory usage: 13.8+ KB


In [101]:
df_gaaraas_final.isnull().sum()

,0
Marque,0
Modele,0
Annee,1
Prix,1
Kilometrage,2
Boite_vitesse,0
Region,0


In [102]:
df_gaaraas_final.head()

,Marque,Modele,Annee,Prix,Kilometrage,Boite_vitesse,Region
0,Nissan,Murano,2008,3200000.0,115305.0,Automatique,Dakar
1,Ford,Escort,2007,2700000.0,174000.0,Manuelle,Dakar
2,Nissan,Qashqai,2014,7800000.0,63000.0,Automatique,Dakar
3,Mazda,6,2010,4000000.0,182000.0,Automatique,Dakar
4,Peugeot,307,2007,2400000.0,160000.0,Manuelle,Dakar


### Conclusion du nettoyage des données Gaaraas

Après la collecte, les données ont été vérifiées et nettoyées afin de faciliter leur utilisation pour les analyses.

Les variables Annee, Prix et Kilometrage ont été converties en variables numériques. Les valeurs particulières rencontrées pendant le nettoyage ont également été identifiées et traitées sans supprimer les annonces concernées.

Le DataFrame final contient 245 observations et 7 variables. Il reste une valeur manquante pour Annee, une pour Prix et deux pour Kilometrage. Ces valeurs correspondent à des informations qui n'étaient pas renseignées dans les annonces d'origine.

Les autres variables ne présentent pas de valeurs manquantes.

In [ ]:
# Sauvegarde des deux jeux de données nettoyés

df_final.to_csv("books_clean.csv", index=False)
df_gaaraas_final.to_csv("gaaraas_clean.csv", index=False)

In [ ]:
from google.colab import files

files.download("books_clean.csv")
files.download("gaaraas_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [103]:
df_final['Prix'] = (
    df_final['Prix']
    .str.replace('£', '', regex=False)
    .astype(float)
)

In [104]:
df_final['Prix'].dtype

dtype('float64')

In [105]:
df_final['Prix'].head()

,Prix
0,51.77
1,53.74
2,50.10
3,47.82
4,54.23


In [106]:
df_final.to_csv("books_clean.csv", index=False)

In [107]:
from google.colab import files
files.download("books_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>